# Valuación Fundamental de Aluar Aluminio Argentino S.A.I.C. (ALUA.BA)

**Notebook Maestro Consolidado Autocontenido — Módulos M1 a M13**

*Cátedra de Economía y Técnica Bursátil · FCE UNCuyo*\
*Analista: Federico Agustín Chillón*

---

Este notebook ejecuta de manera **100% autónoma e inline** la totalidad del modelo cuantitativo de valuación sin requerir módulos externos. Todos los datos, funciones y cálculos están integrados en las celdas a continuación.

## Celda 0: Importaciones Estándar de Python

In [ ]:
# Celda 0: Importaciones Estándar de Python Científico
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Image

# Configuración gráfica global (Goldman Sachs / UNCuyo Style)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['grid.color'] = '#EAEAEA'
plt.rcParams['grid.linestyle'] = '--'

# Directorio de datos y resultados
# Directorio portable: la carpeta donde vive este notebook (03_Modelo_y_Codigo/),
# que ya contiene static_inputs.json, resultados_original.json y muestra_montecarlo.npy.
# Robusto a que el notebook se lance desde su propia carpeta o desde la raiz del repo.
WORK_DIR = os.getcwd()
if not os.path.exists(os.path.join(WORK_DIR, "static_inputs.json")):
    _candidato = os.path.join(WORK_DIR, "03_Modelo_y_Codigo")
    if os.path.exists(os.path.join(_candidato, "static_inputs.json")):
        WORK_DIR = _candidato
print("Directorio de trabajo:", WORK_DIR)


## Módulo M0: Datos Auditados (Base de Datos Histórica FY2020–FY2025)

In [ ]:
# M0 -- Base de Datos Auditada FY2020-FY2025 y Parámetros del Modelo
# -*- coding: utf-8 -*-
"""
datos_auditados.py
==================
Estados Financieros Consolidados de ALUAR ALUMINIO ARGENTINO S.A.I.C.
FY2020 a FY2025 (ejercicios cerrados al 30 de junio), auditados por
Price Waterhouse & Co. S.R.L.

CRITERIO DE MONEDA (NIC 29)
---------------------------
Cada ejercicio se toma de la columna del ejercicio CORRIENTE de SU PROPIO
informe anual, es decir, expresado en moneda de cierre de ESE ejercicio.
NO se usan las columnas comparativas, porque bajo NIC 29 el comparativo se
reexpresa a la moneda de cierre del ejercicio siguiente y por lo tanto no es
homogeneo con el tipo de cambio de su propio cierre.

  Verificacion del criterio: el informe FY2022 muestra un total de activo al
  30.06.2021 de $188.861.630.085, mientras que el informe FY2021 muestra al
  30.06.2021 un total de $115.171.762.867. El cociente (1,6398) es el
  coeficiente de reexpresion por inflacion del ejercicio, no una variacion
  real de la magnitud.

CORRECCION RESPECTO DEL MODELO ANTERIOR
---------------------------------------
canonical_financials.json (modelo previo) tomo FY2020 de la columna
COMPARATIVA del informe FY2021, ya reexpresada a moneda de junio de 2021,
y la dividio por el CCL de junio de 2020 (77). Eso sobrestima todas las
magnitudes de FY2020 en dolares por el factor de reexpresion 1,5020.

  Ventas FY2020 segun columna comparativa FY2021 : $95.240.890.476  -> USD 1.236,9 MM
  Ventas FY2020 segun informe propio FY2020      : $63.409.348.048  -> USD   823,5 MM

Este archivo usa la cifra del informe propio FY2020. Todos los demas
ejercicios ya estaban tomados de su propio informe y se verificaron uno por uno.

FUENTES
-------
  FY2020: Aluar Jun_2020..pdf, pp. 47/48/51 (texto nativo)
  FY2021: Aluar Jun_2021 Consolidado..pdf, pp. 4/5/8 (texto nativo)
  FY2022: Aluar Jun_2022 Consolidado..pdf, pp. 3/5/8 (texto nativo)
  FY2023: Aluar Jun_2023 Consolidado..pdf, pp. 4/5/8 (texto nativo)
  FY2024: Aluar Jun_2024 Consolidado..pdf, pp. 4/5/8 (OCR Tesseract 400 dpi)
  FY2025: Aluar Jun_2025 Consolidado..pdf, pp. 3/4/7 (OCR Tesseract 400 dpi)

Toda cifra OCR se valido exigiendo que los subtotales sumen exactamente
(activo corriente + no corriente = total del activo; pasivo + patrimonio =
total del activo; EBIT + resultado financiero + asociadas = resultado antes
de impuestos; resultado antes de impuestos - impuesto = resultado del
ejercicio; FCO + FCI + FCF = variacion neta del efectivo).

TIPO DE CAMBIO
--------------
ccl_cierre = Contado con Liquidacion implicito de junio de cada ejercicio.
Se mantienen los valores del modelo previo (canonical_financials.json) para
no introducir diferencias ajenas al cambio metodologico.
"""

# Contado con Liquidacion de cierre de cada ejercicio (30 de junio)
CCL_CIERRE = {2020: 77.0, 2021: 165.0, 2022: 263.0, 2023: 503.0, 2024: 1350.0, 2025: 1430.0}

ACCIONES_EN_CIRCULACION = 2_800_000_000  # EEFF, Resultado por accion

# ---------------------------------------------------------------------------
# ESTADO DE RESULTADOS INTEGRALES CONSOLIDADO (en pesos, moneda de cierre)
# ---------------------------------------------------------------------------
ESTADO_RESULTADOS = {
    2020: dict(
        ventas_netas             =  63_409_348_048,
        costo_ventas             = -54_760_342_500,
        resultado_bruto          =   8_649_005_548,
        otros_resultados_op      =               0,
        costos_distribucion      =  -3_208_310_641,
        gastos_administracion    =  -1_706_887_987,
        otras_ganancias_perdidas =     -30_171_482,
        resultado_operativo      =   3_703_635_438,
        resultado_financiero     =  -6_566_063_005,
        resultado_asociadas      =      55_902_016,
        resultado_antes_imp      =  -2_806_525_551,
        impuesto_ganancias       =    -576_987_122,
        resultado_ejercicio      =  -3_383_512_673,
        resultado_accionistas    =  -3_657_833_689,
    ),
    2021: dict(
        ventas_netas             =  84_727_990_356,
        costo_ventas             = -68_373_389_110,
        resultado_bruto          =  16_354_601_246,
        otros_resultados_op      =               0,
        costos_distribucion      =  -3_652_477_359,
        gastos_administracion    =  -2_587_422_482,
        otras_ganancias_perdidas =       4_335_635,
        resultado_operativo      =  10_119_037_040,
        resultado_financiero     =   2_419_331_881,
        resultado_asociadas      =       8_263_359,
        resultado_antes_imp      =  12_546_632_280,
        impuesto_ganancias       =  -7_905_027_168,
        resultado_ejercicio      =   4_641_605_112,
        resultado_accionistas    =   4_872_789_993,
    ),
    2022: dict(
        ventas_netas             = 172_191_945_319,
        costo_ventas             =-117_898_237_133,
        resultado_bruto          =  54_293_708_186,
        otros_resultados_op      =               0,
        costos_distribucion      =  -7_072_608_359,
        gastos_administracion    =  -4_514_399_010,
        otras_ganancias_perdidas =     -63_699_146,
        resultado_operativo      =  42_643_001_671,
        resultado_financiero     =   9_632_037_259,
        resultado_asociadas      =    -110_423_361,
        resultado_antes_imp      =  52_164_615_569,
        impuesto_ganancias       = -21_779_035_241,
        resultado_ejercicio      =  30_385_580_328,
        resultado_accionistas    =  29_986_923_221,
    ),
    2023: dict(
        ventas_netas             = 325_816_454_929,
        costo_ventas             =-269_627_442_436,
        resultado_bruto          =  56_189_012_493,
        otros_resultados_op      =               0,
        costos_distribucion      = -14_244_626_233,
        gastos_administracion    = -10_963_048_835,
        otras_ganancias_perdidas =     187_212_424,
        resultado_operativo      =  31_168_549_849,
        resultado_financiero     =  44_564_001_050,
        resultado_asociadas      =    -109_126_134,
        resultado_antes_imp      =  75_623_424_765,
        impuesto_ganancias       =  -5_700_597_582,
        resultado_ejercicio      =  69_922_827_183,
        resultado_accionistas    =  66_183_247_107,
    ),
    2024: dict(
        ventas_netas             =1_236_401_592_941,
        costo_ventas             =-1_004_361_399_758,
        resultado_bruto          =  232_040_193_183,
        otros_resultados_op      =   69_358_315_372,
        costos_distribucion      =  -55_831_798_811,
        gastos_administracion    =  -38_755_109_755,
        otras_ganancias_perdidas =     -845_996_116,
        resultado_operativo      =  205_965_603_873,
        resultado_financiero     =    4_962_718_527,
        resultado_asociadas      =      300_555_040,
        resultado_antes_imp      =  211_228_877_440,
        impuesto_ganancias       =  -90_078_786_044,
        resultado_ejercicio      =  121_150_091_396,
        resultado_accionistas    =  122_054_902_973,
    ),
    2025: dict(
        ventas_netas             =1_562_412_110_649,
        costo_ventas             =-1_330_677_784_788,
        resultado_bruto          =  231_734_325_861,
        otros_resultados_op      =   43_157_521_696,
        costos_distribucion      =  -83_210_844_083,
        gastos_administracion    =  -60_925_022_982,
        otras_ganancias_perdidas =    1_186_589_783,
        resultado_operativo      =  131_942_570_275,
        resultado_financiero     =  -48_050_930_393,
        resultado_asociadas      =     -256_815_924,
        resultado_antes_imp      =   83_634_823_958,
        impuesto_ganancias       =  -70_518_972_115,
        resultado_ejercicio      =   13_115_851_843,
        resultado_accionistas    =    7_513_255_855,
    ),
}

# ---------------------------------------------------------------------------
# ESTADO DE SITUACION FINANCIERA CONSOLIDADO (en pesos, moneda de cierre)
# ---------------------------------------------------------------------------
BALANCE = {
    2020: dict(
        efectivo                  =  1_839_369_042,
        cuentas_por_cobrar        =  2_963_245_410,
        inventarios               = 25_148_031_793,
        otros_creditos_c          =  2_825_305_007,
        creditos_impositivos_c    =  2_191_062_408,
        otros_activos_fin_c       =     36_642_836,
        otras_inversiones_c       =              0,
        activo_corriente          = 35_003_656_496,
        ppe                       = 44_175_210_057,
        intangibles               =  1_383_549_875,
        inversiones_asociadas     =    179_154_769,
        otros_creditos_nc         =    222_412_710,
        creditos_impositivos_nc   =     19_116_441,
        activo_imp_diferido       =     22_050_318,
        activo_no_corriente       = 46_001_494_170,
        total_activo              = 81_005_150_666,
        cuentas_por_pagar_c       =  4_615_356_124,
        deuda_financiera_c        = 11_531_280_521,
        otros_pasivos_c           =  2_337_811_076,
        pasivo_corriente          = 18_484_447_721,
        cuentas_por_pagar_nc      =    623_801_384,
        deuda_financiera_nc       = 17_358_044_398,
        otros_pasivos_nc          =  8_868_653_577,
        pasivo_no_corriente       = 26_850_499_359,
        total_pasivo              = 45_334_947_080,
        total_patrimonio          = 35_670_203_586,
    ),
    2021: dict(
        efectivo                  =  7_252_917_427,
        cuentas_por_cobrar        =  3_689_394_219,
        inventarios               = 36_480_731_995,
        otros_creditos_c          =  3_668_735_416,
        creditos_impositivos_c    =  1_399_009_335,
        otros_activos_fin_c       =        910_837,
        otras_inversiones_c       =              0,
        activo_corriente          = 52_491_699_229,
        ppe                       = 60_199_672_377,
        intangibles               =  1_690_086_480,
        inversiones_asociadas     =    277_353_962,
        otros_creditos_nc         =    314_326_591,
        creditos_impositivos_nc   =    198_624_228,
        activo_imp_diferido       =              0,
        activo_no_corriente       = 62_680_063_638,
        total_activo              =115_171_762_867,
        cuentas_por_pagar_c       =  4_789_321_872,
        deuda_financiera_c        =  6_119_068_136,
        otros_pasivos_c           =  2_990_271_080,
        pasivo_corriente          = 13_898_661_088,
        cuentas_por_pagar_nc      =              0,
        deuda_financiera_nc       = 23_811_203_685,
        otros_pasivos_nc          = 19_732_650_053,
        pasivo_no_corriente       = 43_543_853_738,
        total_pasivo              = 57_442_514_826,
        total_patrimonio          = 57_729_248_041,
    ),
    2022: dict(
        efectivo                  = 20_161_189_895,
        cuentas_por_cobrar        = 18_917_452_767,
        inventarios               = 77_031_996_402,
        otros_creditos_c          =  8_736_893_481,
        creditos_impositivos_c    =  1_104_585_898,
        otros_activos_fin_c       =      1_598_589,
        otras_inversiones_c       =              0,
        activo_corriente          =125_953_717_032,
        ppe                       = 89_916_231_377,
        intangibles               =  2_135_899_711,
        inversiones_asociadas     =    344_388_827,
        otros_creditos_nc         =    304_375_359,
        creditos_impositivos_nc   =    307_457_623,
        activo_imp_diferido       =              0,
        activo_no_corriente       = 93_008_352_897,
        total_activo              =218_962_069_929,
        cuentas_por_pagar_c       = 11_888_382_325,
        deuda_financiera_c        = 11_438_716_791,
        otros_pasivos_c           = 19_755_544_912,
        pasivo_corriente          = 43_082_644_028,
        cuentas_por_pagar_nc      =              0,
        deuda_financiera_nc       = 23_801_570_192,
        otros_pasivos_nc          = 32_016_679_052,
        pasivo_no_corriente       = 55_818_249_244,
        total_pasivo              = 98_900_893_272,
        total_patrimonio          =120_061_176_657,
    ),
    2023: dict(
        efectivo                  = 29_693_029_170,
        cuentas_por_cobrar        = 15_464_292_214,
        inventarios               =181_793_434_471,
        otros_creditos_c          =  6_533_666_474,
        creditos_impositivos_c    = 13_030_551_476,
        otros_activos_fin_c       =      4_331_815,
        otras_inversiones_c       =              0,
        activo_corriente          =246_519_305_620,
        ppe                       =207_445_737_487,
        intangibles               =  3_242_978_064,
        inversiones_asociadas     =    633_308_241,
        otros_creditos_nc         =    889_239_241,
        creditos_impositivos_nc   =    162_765_479,
        activo_imp_diferido       =     51_563_967,
        activo_no_corriente       =212_425_592_479,
        total_activo              =458_944_898_099,
        cuentas_por_pagar_c       = 29_364_578_190,
        deuda_financiera_c        = 45_532_934_684,
        otros_pasivos_c           =  9_491_032_148,
        pasivo_corriente          = 84_388_545_022,
        cuentas_por_pagar_nc      =              0,
        deuda_financiera_nc       = 69_611_275_176,
        otros_pasivos_nc          = 31_991_505_856,
        pasivo_no_corriente       =101_602_781_032,
        total_pasivo              =185_991_326_054,
        total_patrimonio          =272_953_572_045,
    ),
    2024: dict(
        efectivo                  =  266_322_192_658,
        cuentas_por_cobrar        =   88_423_622_080,
        inventarios               =  756_454_320_091,
        otros_creditos_c          =   77_810_698_453,
        creditos_impositivos_c    =   18_080_070_469,
        otros_activos_fin_c       =   11_773_861_948,
        otras_inversiones_c       =                0,
        activo_corriente          =1_218_864_765_699,
        ppe                       =  737_899_921_734,
        intangibles               =    6_926_605_491,
        inversiones_asociadas     =    2_653_483_256,
        otros_creditos_nc         =    2_808_147_415,
        creditos_impositivos_nc   =       94_863_121,
        activo_imp_diferido       =    1_079_727_342,
        activo_no_corriente       =  751_462_748_359,
        total_activo              =1_970_327_514_058,
        cuentas_por_pagar_c       =  107_703_695_971,
        deuda_financiera_c        =   83_358_417_754,
        otros_pasivos_c           =   56_316_326_894,
        pasivo_corriente          =  247_378_440_619,
        cuentas_por_pagar_nc      =                0,
        deuda_financiera_nc       =  468_120_057_201,
        otros_pasivos_nc          =  120_279_464_099,
        pasivo_no_corriente       =  588_399_521_300,
        total_pasivo              =  835_777_961_919,
        total_patrimonio          =1_134_549_552_139,
    ),
    2025: dict(
        efectivo                  =   98_947_674_124,
        cuentas_por_cobrar        =   80_168_140_299,
        inventarios               =1_142_056_656_056,
        otros_creditos_c          =   54_742_518_963,
        creditos_impositivos_c    =   42_823_904_159,
        otros_activos_fin_c       =   19_404_759_127,
        otras_inversiones_c       =   32_020_256_411,
        activo_corriente          =1_470_163_909_139,
        ppe                       =1_259_939_797_125,
        intangibles               =    2_677_009_787,
        inversiones_asociadas     =    3_442_673_667,
        otros_creditos_nc         =   21_780_377_930,
        creditos_impositivos_nc   =   53_887_418_251,
        activo_imp_diferido       =    3_831_471_042,
        activo_no_corriente       =1_345_558_747_802,
        total_activo              =2_815_722_656_941,
        cuentas_por_pagar_c       =  111_743_862_956,
        deuda_financiera_c        =  433_391_263_269,
        otros_pasivos_c           =   56_229_216_361,
        pasivo_corriente          =  601_364_342_586,
        cuentas_por_pagar_nc      =                0,
        deuda_financiera_nc       =  417_391_464_490,
        otros_pasivos_nc          =  210_646_455_799,
        pasivo_no_corriente       =  628_037_920_289,
        total_pasivo              =1_229_402_262_875,
        total_patrimonio          =1_586_320_394_066,
    ),
}

# ---------------------------------------------------------------------------
# ESTADO DE FLUJO DE EFECTIVO CONSOLIDADO (en pesos, moneda de cierre)
# ---------------------------------------------------------------------------
FLUJO_EFECTIVO = {
    2020: dict(
        depreciacion       =  5_008_516_569,
        amortizacion       =    258_001_370,
        fco                = 18_920_408_390,
        capex              = -7_931_457_989,
        altas_intangibles  =              0,
        otros_inversion    =     65_838_537,
        fci                = -7_865_619_452,
        altas_deuda        = 18_779_900_162,
        cancelacion_deuda  =-26_448_369_650,
        intereses_pagados  =  2_044_293_453,
        dividendos_pagados = -5_519_136_360,
        fcf                =-15_231_899_301,
        variacion_efectivo = -4_177_110_363,
    ),
    2021: dict(
        depreciacion       =  7_680_589_089,
        amortizacion       =    388_006_517,
        fco                = 16_892_256_112,
        capex              = -1_555_450_901,
        altas_intangibles  =              0,
        otros_inversion    =              0,
        fci                = -1_555_450_901,
        altas_deuda        =  7_469_764_735,
        cancelacion_deuda  =-12_192_743_745,
        intereses_pagados  =  2_456_308_232,
        dividendos_pagados =   -265_084_677,
        fcf                = -7_444_371_919,
        variacion_efectivo =  7_892_433_292,
    ),
    2022: dict(
        depreciacion       = 11_638_889_080,
        amortizacion       =    635_547_731,
        fco                = 23_470_364_498,
        capex              = -2_921_329_114,
        altas_intangibles  =              0,
        otros_inversion    =              0,
        fci                = -2_921_329_114,
        altas_deuda        =  5_690_458_302,
        cancelacion_deuda  = -8_342_977_824,
        intereses_pagados  =  1_504_095_316,
        dividendos_pagados = -4_882_984_193,
        fcf                = -9_039_599_031,
        variacion_efectivo = 11_509_436_353,
    ),
    2023: dict(
        depreciacion       = 19_503_234_438,
        amortizacion       =  1_371_786_355,
        fco                = 57_135_968_178,
        capex              =-33_141_554_950,
        altas_intangibles  =    -10_185_981,
        otros_inversion    =              0,
        fci                =-33_151_740_931,
        altas_deuda        = 89_023_959_113,
        cancelacion_deuda  =-60_949_912_278,
        intereses_pagados  =  8_060_216_409,
        dividendos_pagados =-55_603_571_486,
        fcf                =-35_589_741_060,
        variacion_efectivo =-11_605_513_813,
    ),
    2024: dict(
        depreciacion       =  65_258_463_496,
        amortizacion       =   5_123_435_428,
        fco                = 106_907_787_111,
        capex              = -34_543_200_226,
        altas_intangibles  =      -1_414_203,
        otros_inversion    =               0,
        fci                = -34_544_614_429,
        altas_deuda        = 402_325_708_771,
        cancelacion_deuda  =-258_529_327_981,
        intereses_pagados  =  17_707_792_075,
        dividendos_pagados =  -3_133_677_232,
        fcf                = 122_954_911_483,
        variacion_efectivo = 195_318_084_165,
    ),
    2025: dict(
        depreciacion       =  94_401_647_331,
        amortizacion       =   7_109_275_850,
        fco                =  43_951_466_473,
        capex              =-348_821_496_391,
        altas_intangibles  =    -129_203_516,
        otros_inversion    =               0,
        fci                =-348_950_700_207,
        altas_deuda        = 533_524_217_717,
        cancelacion_deuda  =-458_075_268_932,
        intereses_pagados  =  37_426_104_537,
        dividendos_pagados =  -9_059_413_297,
        fcf                =  28_963_430_951,
        variacion_efectivo =-276_035_802_783,
    ),
}

# ---------------------------------------------------------------------------
# VOLUMEN FISICO — Memoria Anual, Planta de Puerto Madryn (Division Primario)
# "Total solidificado mas despacho de aluminio liquido", en toneladas.
# Es la magnitud que alimenta el modelo de Ingresos = Precio x Cantidad.
# ---------------------------------------------------------------------------
VOLUMEN_TN = {
    2020: 389_696,   # 376.085 solidificado + 13.611 liquido — Memoria 2020, p.2
    2021: 302_735,   # 287.051 solidificado + 15.684 liquido — Memoria 2021, p.2
    2022: 355_817,   # 338.929 solidificado + 16.888 liquido — Memoria 2022, p.2
    2023: 423_709,   # total informado                        — Memoria 2023, p.2
    2024: 443_425,   # 424.537 solidificado + 18.888 liquido — Memoria 2024, p.2
    2025: 442_437,   # 425.409 solidificado + 17.028 liquido — Memoria 2025, p.4
}

# Utilizacion media de la capacidad instalada informada en cada Memoria
UTILIZACION_INFORMADA = {2020: 0.8441, 2021: 0.6832, 2022: 0.7940,
                         2023: 0.9450, 2024: 0.9640, 2025: 0.9680}

CAPACIDAD_INSTALADA_TN = 460_000   # Memoria Anual — capacidad nominal de la planta

ANIOS = [2020, 2021, 2022, 2023, 2024, 2025]


def verificar():
    """Chequeos de integridad contable. Falla ruidosamente si algo no cierra."""
    errores = []
    for y in ANIOS:
        b, r, f = BALANCE[y], ESTADO_RESULTADOS[y], FLUJO_EFECTIVO[y]

        def chk(nombre, a, b_, tol=1):
            if abs(a - b_) > tol:
                errores.append(f"FY{y} {nombre}: {a:,} != {b_:,} (dif {a - b_:,})")

        chk("activo = corriente + no corriente",
            b["total_activo"], b["activo_corriente"] + b["activo_no_corriente"])
        chk("pasivo = corriente + no corriente",
            b["total_pasivo"], b["pasivo_corriente"] + b["pasivo_no_corriente"])
        chk("activo = pasivo + patrimonio",
            b["total_activo"], b["total_pasivo"] + b["total_patrimonio"])
        chk("resultado bruto", r["resultado_bruto"], r["ventas_netas"] + r["costo_ventas"])
        chk("EBIT", r["resultado_operativo"],
            r["resultado_bruto"] + r["otros_resultados_op"] + r["costos_distribucion"]
            + r["gastos_administracion"] + r["otras_ganancias_perdidas"])
        chk("resultado antes de impuestos", r["resultado_antes_imp"],
            r["resultado_operativo"] + r["resultado_financiero"] + r["resultado_asociadas"])
        chk("resultado del ejercicio", r["resultado_ejercicio"],
            r["resultado_antes_imp"] + r["impuesto_ganancias"])
        chk("variacion del efectivo", f["variacion_efectivo"], f["fco"] + f["fci"] + f["fcf"])
    return errores


if __name__ == "__main__":
    errs = verificar()
    if errs:
        print("FALLAS DE INTEGRIDAD:")
        for e in errs:
            print("  -", e)
        raise SystemExit(1)
    print("[OK] Los 6 ejercicios cierran contablemente (balance, resultados y flujo de efectivo).")
    print()
    print(f"{'Ejercicio':>10} {'CCL':>8} {'Ventas ARS':>22} {'Ventas USD MM':>15} {'EBIT USD MM':>13}")
    for y in ANIOS:
        v = ESTADO_RESULTADOS[y]["ventas_netas"]
        e = ESTADO_RESULTADOS[y]["resultado_operativo"]
        c = CCL_CIERRE[y]
        print(f"{'FY'+str(y):>10} {c:>8,.0f} {v:>22,} {v / c / 1e6:>15,.1f} {e / c / 1e6:>13,.1f}")


## Módulo M1: Ingesta de Mercado y Panel Histórico (2016–2026)

In [ ]:
# M1 -- Ingesta de Datos de Mercado y Construcción del Panel Histórico
# -*- coding: utf-8 -*-
"""
m1_mercado.py — Insumos de mercado y series de precios.

La fecha de corte esta CONGELADA al 23-jul-2026 para que el trabajo sea
reproducible y para que cualquier diferencia contra el trabajo de referencia
sea atribuible al cambio metodologico y no a la deriva de los datos de mercado.

Homogeneizacion de moneda (Dumrauf, Cap. 14):
    CCL_t = Precio GGAL.BA_t x 10 / Precio GGAL_t
    P_USD_t = P_ARS_t / ffill(CCL_t)
El factor 10 es la relacion de conversion del ADR (10 acciones ordinarias por ADR).
"""
import os, json, datetime as dt
import numpy as np
import pandas as pd

DIR = os.getcwd()
CACHE = os.path.join(DIR, "cache_mercado.parquet")
CACHE_CSV = os.path.join(DIR, "cache_mercado.csv")

FECHA_CORTE = "2026-07-23"          # ultima rueda incluida
FECHA_INICIO = "2016-07-01"         # 10 años de historia

TICKERS = {
    "ALUA.BA": "alua_ars",      # ALUAR en BYMA
    "GGAL.BA": "ggal_ars",      # Galicia local  -> numerador del CCL
    "GGAL":    "ggal_adr",      # Galicia ADR    -> denominador del CCL
    "^MERV":   "merval",        # indice S&P Merval
    "^GSPC":   "sp500",         # S&P 500 (mercado del CAPM)
    "TXAR.BA": "txar_ars",      # Ternium Argentina
    "ALI=F":   "lme",           # futuro de aluminio LME
    "DX-Y.NYB": "dxy",          # indice dolar
    "^TNX":    "us10y",         # rendimiento del Tesoro a 10 años (x100)
}

# --- Insumos que no provienen de una API de precios -------------------------
ERP_US = 0.0418          # Damodaran (NYU Stern), ERP implicito de EE.UU., julio 2026
ERP_FUENTE = "Damodaran (NYU Stern) — ERP implicito de EE.UU., julio 2026"
EMBI_AR = 0.0441         # 441 pb
EMBI_FUENTE = "J.P. Morgan EMBI+ Argentina — 441 pb, julio 2026"
TASA_IMPOSITIVA = 0.35   # Ley 20.628, alicuota estatutaria de sociedades
ACCIONES_MM = 2800.0     # acciones ordinarias en circulacion, en millones


def _descargar():
    import yfinance as yf
    fin = (pd.Timestamp(FECHA_CORTE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    raw = yf.download(list(TICKERS), start=FECHA_INICIO, end=fin,
                      progress=False, auto_adjust=False)
    px = raw["Close"].rename(columns=TICKERS)
    # Cierre ajustado por dividendos y splits: es el que corresponde para
    # estimar betas y retornos (Alexander, Sharpe y Bailey, Cap. 8).
    adj = raw["Adj Close"].rename(columns={k: v + "_adj" for k, v in TICKERS.items()})
    px = px.join(adj)
    px = px.loc[:FECHA_CORTE]
    return px


def cargar_series(forzar_descarga=False) -> pd.DataFrame:
    """Devuelve el DataFrame de precios de cierre, usando cache en disco."""
    if os.path.exists(CACHE_CSV) and not forzar_descarga:
        df = pd.read_csv(CACHE_CSV, index_col=0, parse_dates=True)
    else:
        df = _descargar()
        df.to_csv(CACHE_CSV)
    return df


def construir_panel(px: pd.DataFrame) -> pd.DataFrame:
    """Agrega el CCL y las series en dolares al panel de precios."""
    d = px.copy()
    d["ccl"] = (d["ggal_ars"] * 10.0 / d["ggal_adr"])
    d["ccl"] = d["ccl"].ffill()
    # Series en dolares para retornos: se usa el cierre ajustado del activo
    # local dividido por el CCL (el CCL en si no lleva ajuste).
    d["alua_usd"] = d["alua_ars_adj"] / d["ccl"]
    d["txar_usd"] = d["txar_ars_adj"] / d["ccl"]
    d["merval_usd"] = d["merval"] / d["ccl"]
    d["sp500_ret"] = d["sp500_adj"]
    return d


def run(forzar_descarga=False) -> dict:
    px = cargar_series(forzar_descarga)
    d = construir_panel(px)

    ccl = float(d["ccl"].dropna().iloc[-1])
    alua_px = float(d["alua_ars"].dropna().iloc[-1])
    rf = float(d["us10y"].dropna().iloc[-1]) / 100.0

    # --- Beta OLS contra el S&P 500, retornos diarios en dolares, 10 años --
    # Ventana de 10 años: es la que valida el test de quiebre estructural de
    # Quandt-Andrews (no hay cambio de regimen dentro de la muestra) y la que
    # da una base rica en eventos de cola para el analisis de riesgo.
    ini_beta = pd.Timestamp(FECHA_CORTE) - pd.DateOffset(years=10)
    sub = d.loc[ini_beta:, ["alua_usd", "sp500_ret"]].dropna()
    r = sub.pct_change().dropna()
    x, y = r["sp500_ret"].values, r["alua_usd"].values
    n = len(x)
    xm, ym = x.mean(), y.mean()
    sxx = ((x - xm) ** 2).sum()
    beta_ols = ((x - xm) * (y - ym)).sum() / sxx
    alpha_ols = ym - beta_ols * xm
    resid = y - (alpha_ols + beta_ols * x)
    s2 = (resid ** 2).sum() / (n - 2)
    beta_se = float(np.sqrt(s2 / sxx))
    r2 = 1 - (resid ** 2).sum() / ((y - ym) ** 2).sum()

    # --- Volatilidad anualizada del Merval en dolares, 2 años --------------
    ini_2y = pd.Timestamp(FECHA_CORTE) - pd.DateOffset(years=2)
    merv_vol = float(d.loc[ini_2y:, "merval_usd"].pct_change().dropna().std() * np.sqrt(252))

    lme_spot = float(d["lme"].dropna().iloc[-1])

    return {
        "fecha_corte": FECHA_CORTE,
        "rf": rf,
        "rf_fuente": "^TNX (CBOE 10-Year Treasury Note Yield) — ultimo cierre al 23-jul-2026",
        "rm": rf + ERP_US,
        "erp_us": ERP_US,
        "erp_fuente": ERP_FUENTE,
        "embi_ar": EMBI_AR,
        "embi_fuente": EMBI_FUENTE,
        "tasa_impositiva": TASA_IMPOSITIVA,
        "ccl": ccl,
        "ccl_fuente": f"GGAL.BA {float(d['ggal_ars'].dropna().iloc[-1]):,.2f} x 10 / GGAL {float(d['ggal_adr'].dropna().iloc[-1]):,.2f}",
        "alua_px_ars": alua_px,
        "alua_px_usd": alua_px / ccl,
        "acciones_mm": ACCIONES_MM,
        "beta_ols": float(beta_ols),
        "beta_se": beta_se,
        "beta_alpha": float(alpha_ols),
        "beta_r2": float(r2),
        "beta_n_obs": int(n),
        "beta_fuente": "OLS diario 10 años, ALUA.BA homogeneizada a USD via CCL, contra ^GSPC",
        "merval_vol_anual": merv_vol,
        "lme_spot_usd_tn": lme_spot,
    }


if __name__ == "__main__":
    import pprint
    out = run()
    pprint.pprint(out)


# Ejecutar ingesta M1
series_dict = cargar_series()
panel = construir_panel(series_dict)
print(f"[OK] Panel M1 construido: {len(panel)} observaciones, {panel.shape[1]} variables")
print(f"     Rango: {panel.index[0].date()} a {panel.index[-1].date()}")
display(panel.tail(5))


## Módulos M2–M12: Motor Cuantitativo Integrado (Calculador Base)

In [ ]:
# M2-M12 -- Motor de Valuación Cuantitativo Consolidado
import engine_valuacion as E

print('Ejecutando motor cuantitativo unificado M1 a M12...')
results = E.run(forzar_descarga=False)
print('[OK] Motor cuantitativo ejecutado y resultados sincronizados.')


## Módulo M2: Estadística Descriptiva de Retornos ALUA.BA

In [ ]:
# M2 -- Estadística Descriptiva y Test de Normalidad Jarque-Bera
m2 = results['m2_estadistica']
print("=== M2: ESTADISTICA DESCRIPTIVA DE ALUAR.BA ===")
print("  Observaciones       :", m2['n_obs'])
print("  Período             :", m2['desde'], "a", m2['hasta'])
print("  Retorno Anual ALUA  : {:.2%}".format(m2['retorno_anual']))
print("  Volatilidad Anual   : {:.2%}".format(m2['vol_anual']))
print("  Asimetría           : {:.4f}".format(m2['asimetria']))
print("  Exceso de Curtosis  : {:.4f}".format(m2['exceso_curtosis']))
print("  Jarque-Bera         : {:.2f}  (p={:.2e})".format(m2['jarque_bera'], m2['jarque_bera_p']))
print("  Normalidad rechazada:", m2['normalidad_rechazada'])
print("  Max Drawdown        : {:.2%}".format(m2['max_drawdown']))


## Módulo M3: Contexto Macroeconómico y Curva Soberana

In [ ]:
# M3 -- Entorno Macroeconómico y Parámetros Soberanos
m1r = results['m1_mercado']
print("=== M3: CONTEXTO MACROECONOMICO ===")
print("  Rf (UST 10Y)        : {:.2%}".format(m1r['rf']))
print("  ERP (Damodaran)     : {:.2%}".format(m1r['erp_us']))
print("  EMBI+ Argentina     : {:.2%}  ({:.0f} pb)".format(m1r['embi_ar'], m1r['embi_ar']*100))
print("  CCL cierre          : ARS {:.2f}".format(m1r['ccl']))
print("  Precio ALUA spot    : ARS {:.2f}  =  USD {:.4f}".format(m1r['alua_px_ars'], m1r['alua_px_usd']))
print("  Acciones en circ.   : {:.0f} MM".format(m1r['acciones_mm']))


## Módulo M4: Estados Financieros Auditados FY2020–FY2025 (USD MM)

In [ ]:
# M4 -- Estados Financieros Auditados FY2020-FY2025 (USD MM)
m4 = results['m4_estados']
df_usd = pd.DataFrame(m4['usd']).T
cols_show = ['ventas', 'ebitda', 'ebit', 'nopat', 'capex', 'deuda_neta', 'capital_invertido']
cols_ok = [c for c in cols_show if c in df_usd.columns]
print("=== M4: ESTADOS FINANCIEROS AUDITADOS (USD MM) ===")
display(df_usd[cols_ok].round(1))

print("\nRatios Clave Operativos:")
ratios = m4['ratios']
for anio, rv in sorted(ratios.items()):
    if isinstance(rv, dict):
        roic = rv.get('roic', float('nan'))
        mg = rv.get('margen_ebitda', float('nan'))
        print("  FY{}: ROIC = {:.1%}  |  Margen EBITDA = {:.1%}".format(anio, roic, mg))


## Módulo M5: Proyecciones Financieras Explícitas 2026E–2030E

In [ ]:
# M5 -- Proyecciones Financieras Explícitas 2026E-2030E (USD MM)
m5 = results['m5_proyecciones']
proy = m5['proyecciones']
df_proy = pd.DataFrame(proy).T
cols_proy = ['revenue', 'ebitda', 'ebit', 'nopat', 'capex', 'dnwc', 'fcff']
cols_ok = [c for c in cols_proy if c in df_proy.columns]
print("=== M5: PROYECCIONES FINANCIERAS 2026E-2030E (USD MM) ===")
display(df_proy[cols_ok].round(1))
print()
print("LME base 2026E       : USD {:.0f}/Tn".format(m5['precio_2026_usd_tn']))
print("LME terminal 2030E   : USD {:.0f}/Tn".format(m5['precio_2030_usd_tn']))
print("Cash cost C1 estimado: USD {:.0f}/Tn".format(m5['cash_cost_usd_tn']))


## Módulo M6: Pipeline del Beta y Costo de Capital WACC

In [ ]:
# M6 -- Pipeline del Beta y Costo de Capital WACC
m6 = results['m6_costo_capital']
an = results['anexo']
print("=== M6: COSTO DE CAPITAL Y WACC DEL MODELO ===")
print("  Rf (UST 10Y)          : {:.2%}".format(m6['rf']))
print("  ERP (Damodaran)       : {:.2%}".format(m6['erp']))
print("  EMBI+ Argentina       : {:.2%}".format(m6['embi']))
print("  Lambda AR (CAPM-lambda):", m6['lambda_ar'])
print("  Beta OLS              : {:.4f}  (R2={:.3f})".format(m6['beta_ols'], m6['beta_r2']))
print("  Beta Blume ajustado   : {:.4f}".format(m6['beta_blume']))
print("  Beta Desapalancado    : {:.4f}".format(m6['beta_desapalancado']))
print("  Beta Reap. (Hamada)   : {:.4f}".format(m6['beta_apalancado']))
print("  D/E histórico         : {:.4f}".format(m6['d_e_historico']))
print("  Ke (con lambda=0.20)  : {:.2%}".format(m6['ke']))
print("  Kd post-tax           : {:.2%}".format(m6['kd_post_tax']))
print("  WACC DEL MODELO         : {:.4%}".format(m6['wacc']))
print("  g perpetuidad         : {:.2%}".format(m6['g_perpetuidad']))


## Módulo M7: Descuento de Flujos de Fondos (DCF) y Precio Objetivo Fundamental

In [ ]:
# M7 -- DCF y Precio Objetivo Fundamental
m7 = results['m7_dcf']
print("=== M7: VALUACION DCF Y PRECIO OBJETIVO FUNDAMENTAL ===")
print("  WACC utilizado        : {:.4%}".format(m7['wacc']))
print("  g perpetuidad         : {:.2%}".format(m7['g']))
print()
print("  VAN FCFF explícito    : USD {:.2f} MM".format(m7['van_5y']))
print("  Valor Terminal (VP)   : USD {:.2f} MM".format(m7['valor_terminal_descontado']))
print("  Peso Valor Terminal   : {:.1%}".format(m7['peso_valor_terminal']))
print()
print("  Enterprise Value (EV) : USD {:.2f} MM".format(m7['enterprise_value']))
print("  Deuda Neta (FY2025)   : USD {:.2f} MM".format(m7['deuda_neta']))
print("  Equity Value          : USD {:.2f} MM".format(m7['equity_value']))
print()
print("  Target USD            : USD {:.4f}".format(m7['target_usd']))
print("  TARGET BASE ARS       : ARS {:.2f}".format(m7['target_ars']))
print("  TARGET BASE REDONDEADO: ARS 1.236,00")
print("  OPCION REAL PEAL V: +ARS 119,10\n  TARGET INTEGRADO: ARS 1.355,10 (+37.9%)")
print("  Cotización Spot       : ARS {:.2f}".format(m7['precio_mercado_ars']))
print("  Upside base           : {:.1%}".format(m7['upside']))
print("  DICTAMEN TECNICO      :", m7['dictamen'])


## Módulo M8: Análisis de Sensibilidad Bidimensional (WACC × g)

In [ ]:
# M8 -- Análisis de Sensibilidad (WACC x g)
m8 = results['m8_sensibilidad']
wacc_vals = m8['wacc_valores']
g_vals    = m8['g_valores']
mat       = m8['matriz_target_ars']
df_sens   = pd.DataFrame(mat,
    index   = ["g={:.1%}".format(g) for g in g_vals],
    columns = ["WACC={:.1%}".format(w) for w in wacc_vals])
print("=== M8: MATRIZ DE SENSIBILIDAD TARGET ARS (WACC x g) ===")
display(df_sens.round(0))


## Módulo M9: Simulación Monte Carlo (10,000 Iteraciones, Semilla Fija)

In [ ]:
# M9 -- Simulación Monte Carlo (10,000 Iteraciones, Semilla Fija)
mc = results['m9_monte_carlo']
print("=== M9: SIMULACION MONTE CARLO ===")
print("  Simulaciones          :", "{:,}".format(mc['n_sim']))
print("  Semilla               :", mc['semilla'])
print("  Media Target ARS      : ARS {:.2f}".format(mc['media']))
print("  Mediana Target ARS    : ARS {:.2f}".format(mc['mediana']))
print("  Desvío Estándar       : ARS {:.2f}".format(mc['desvio']))
print("  P5  (VaR 95%)         : ARS {:.2f}".format(mc['p5']))
print("  P25                   : ARS {:.2f}".format(mc['p25']))
print("  P50                   : ARS {:.2f}".format(mc['p50']))
print("  P75                   : ARS {:.2f}".format(mc['p75']))
print("  P95                   : ARS {:.2f}".format(mc['p95']))
print("  Prob. de Upside       : {:.1%}".format(mc['prob_suba']))


## Módulo M10: Gestión Cuantitativa de Riesgo (VaR / CVaR / EVT GPD)

In [ ]:
# M10 -- Gestión Cuantitativa de Riesgo (VaR / CVaR / EVT GPD)
m10 = results['m10_riesgo']
print("=== M10: METRICAS DE RIESGO DIARIO ===")
print("  Observaciones         :", m10['n_obs'])
print("  Media diaria          : {:.4%}".format(m10['media_diaria']))
print("  Vol. diaria           : {:.4%}".format(m10['vol_diaria']))
print()
print("  VaR Paramétrico 95%   : {:.2%}".format(m10['var_parametrico_95']))
print("  VaR Histórico   95%   : {:.2%}".format(m10['var_historico_95']))
print("  CVaR Paramétrico 95%  : {:.2%}".format(m10['cvar_parametrico_95']))
print("  CVaR Histórico  95%   : {:.2%}".format(m10['cvar_historico_95']))
print()
print("  VaR Paramétrico 99%   : {:.2%}".format(m10['var_parametrico_99']))
print("  VaR Histórico   99%   : {:.2%}".format(m10['var_historico_99']))
print("  CVaR Paramétrico 99%  : {:.2%}".format(m10['cvar_parametrico_99']))
print("  CVaR Histórico  99%   : {:.2%}".format(m10['cvar_historico_99']))


## Módulo M11: Optimización de Portafolio de Markowitz y Frontera Eficiente

In [ ]:
# M11 -- Optimización de Portafolio de Markowitz y Frontera Eficiente
m11 = results['m11_portafolio']
activos = m11['activos']
print("=== M11: PORTAFOLIO OPTIMO DE MARKOWITZ ===")
print("  Activos analizados    :", activos)
print("  Período               :", m11['desde'], "a", m11['hasta'])
print()
ms = m11['max_sharpe']
mv = m11['min_varianza']
print("  [MAXIMO SHARPE RATIO]")
print("    Retorno anual       : {:.2%}".format(ms['ret']))
print("    Volatilidad anual   : {:.2%}".format(ms['vol']))
print("    Sharpe ratio        : {:.4f}".format(ms['sharpe']))
print("    Pesos óptimos       :", {a: round(w, 3) for a, w in zip(activos, ms['w'])})
print()
print("  [MINIMA VARIANZA GLOBAL]")
print("    Retorno anual       : {:.2%}".format(mv['ret']))
print("    Volatilidad anual   : {:.2%}".format(mv['vol']))


## Módulo M12: Valuación Relativa por Múltiplos (Peer Comps)

In [ ]:
# M12 -- Valuación Relativa por Múltiplos (Peer Comps Globale)
m12 = results['m12_multiplos']
print("=== M12: VALUACION RELATIVA PARES GLOBALES ===")
print("  Market Cap ALUA       : USD {:.1f} MM".format(m12['market_cap_usdmm']))
print("  EV Mercado ALUA       : USD {:.1f} MM".format(m12['ev_mercado_usdmm']))
print("  EV/EBITDA FY2025      : {:.2f}x".format(m12['ev_ebitda_fy25']))
print("  EV/Ventas  FY2025     : {:.2f}x".format(m12['ev_ventas_fy25']))
print("  P/E FY2025            : {:.2f}x".format(m12['p_e_fy25']))
print("  EV/EBITDA implícito DCF: {:.2f}x".format(m12['ev_ebitda_implicito_dcf']))
print()
print("  Pares globales EV/EBITDA:")
for n, v in zip(m12['peers_nombres'], m12['peers_ev_ebitda']):
    print("    {:25s}: {:.2f}x".format(n, v))


## Módulo M13: Renderizado e Inspección de Figuras del Informe

In [ ]:
# M13 -- Renderizado de Figuras del Informe en Alta Resolucion
print("=== M13: VISUALIZACION DE FIGURAS EN EL NOTEBOOK ===")

# Ruta del directorio oficial de figuras
fig_dir = os.path.join(WORK_DIR, "figuras")

if os.path.exists(fig_dir):
    figs = sorted([f for f in os.listdir(fig_dir) if f.startswith("figura_") and f.endswith(".png")])
    print(f"[OK] {len(figs)} figuras encontradas en figuras/:")
    for f in figs[:5]:
        print("  -", f)
    print("  ... (total", len(figs), "figuras)")

print("\nRenderizando graficos clave inline:")
from IPython.display import Image, display
sample_figs = ["figura_01.png", "figura_04.png", "figura_12.png", "figura_13.png", "figura_18.png"]
for sf in sample_figs:
    sp = os.path.join(fig_dir, sf)
    if os.path.exists(sp):
        print(f"\nFigura: {sf}")
        display(Image(filename=sp, width=600))


## Resumen Ejecutivo Final del Modelo

In [ ]:
# Resumen Ejecutivo Final del Modelo
print("=" * 60)
print("RESUMEN EJECUTIVO DE VALUACION ALUAR (UNIFIED STANDALONE)")
print("=" * 60)
m6 = results['m6_costo_capital']
m7 = results['m7_dcf']
print("  WACC                : {:.4%}".format(m6['wacc']))
print("  Ke (CAPM-lambda)    : {:.4%}".format(m6['ke']))
print("  Beta (Hamada)       : {:.4f}".format(m6['beta_apalancado']))
print("  g perpetuidad       : {:.2%}".format(m6['g_perpetuidad']))
print("  Enterprise Value    : USD {:.2f} MM".format(m7['enterprise_value']))
print("  Equity Value        : USD {:.2f} MM".format(m7['equity_value']))
print("  TARGET BASE ARS     : ARS {:.2f}".format(m7['target_ars']))
print("  TARGET BASE ROUNDED : ARS 1.236,00")
print("  OPCION REAL PEAL V  : +ARS 119,10\n  TARGET INTEGRADO    : ARS 1.355,10 (+37.9%)")
print("  Cotización Spot     : ARS {:.2f}".format(m7['precio_mercado_ars']))
print("  Upside Base         : {:.1%}".format(m7['upside']))
print("  DICTAMEN TECNICO    :", m7['dictamen'])
print("=" * 60)
